In [1]:
import pandas as pd 

df = pd.read_csv('data.csv')

In [2]:
df.shape

(47113, 984)

In [3]:
x = df[[i for i in df.columns if i not in ['index','canon_smiles'] and 'pIC50' not in i]]
y = df[['herg_pIC50', 'hepg2_pIC50', 'cyp2c9_pIC50','cyp2d6_pIC50', 'cyp3a4_pIC50']]

In [4]:
# https://github.com/YyzHarry/imbalanced-regression
# sts-b-dir 폴더 git clone 하기!
# sts-b-dir 폴더 내에 util.py가 있는데 이거 이름을 utils.py로 바꿔야 밑에 함수가 불러와짐
import sys
from collections import Counter
from scipy.ndimage import convolve1d

sys.path.append(r'D:\\minimax\\toxic_prediction\sts-b-dir') # LDS를 위한 함수를 불러오기 위해서

from utils import get_lds_kernel_window

def get_bin_idx(label, num_bins, label_min, label_max):
    if label_max == label_min:
        return 0
    
    normalized = (label - label_min) / (label_max - label_min)
    bin_idx = int(normalized * num_bins)   
    return max(0, min(num_bins - 1, bin_idx))

def lds(labels):
    # preds, labels: [Ns,], "Ns" is the number of total samples
    # assign each label to its corresponding bin (start from 0)
    # with your defined get_bin_idx(), return bin_index_per_label: [Ns,] 
    label_min = min(labels)
    label_max = max(labels)
    num_bins = 50

    bin_index_per_label = [get_bin_idx(label, num_bins, label_min, label_max) for label in labels]

    # calculate empirical (original) label distribution: [Nb,]
    # "Nb" is the number of bins
    Nb = max(bin_index_per_label) + 1
    num_samples_of_bins = dict(Counter(bin_index_per_label))
    emp_label_dist = [num_samples_of_bins.get(i, 0) for i in range(Nb)]

    # lds_kernel_window: [ks,], here for example, we use gaussian, ks=5, sigma=2
    lds_kernel_window = get_lds_kernel_window(kernel='gaussian', ks=5, sigma=2)
    # calculate effective label distribution: [Nb,]
    eff_label_dist = convolve1d(np.array(emp_label_dist), weights=lds_kernel_window, mode='constant')

    # Use re-weighting based on effective label distribution, sample-wise weights: [Ns,]
    eff_num_per_label = [eff_label_dist[bin_idx] for bin_idx in bin_index_per_label]
    lds_pIC50 = [np.float32(1 / x) for x in eff_num_per_label]
    lds_pIC50 = lds_pIC50 / np.mean(lds_pIC50) # 원본 코드에는 이게 없는데 이렇게 해야 가중치 간 값의 차이가 좀 생김
    # 이거 안하면 0.62, 0.0054 이런식이라 모델이 제대로 가중치 적용을 못하는듯 하여 이렇게 한 줄 추가함

    return lds_pIC50

In [13]:
from sklearn.decomposition import PCA
import joblib

file_path = 'D:\minimax\\toxic_prediction'

# 학습 시
pca = PCA(n_components=0.999)
df_pca_values = pca.fit_transform(x) # 여기서 x는 977개의 feature를 가진 학습 데이터

# 핵심: 학습된 PCA 객체 자체를 저장
joblib.dump(pca, file_path + "\\toxic_pca.pkl")
# (선택사항) 학습 데이터의 컬럼 순서도 저장해야 안전합니다.
joblib.dump(x.columns.tolist(), file_path + "\\toxic_feature_names.pkl")

['D:\\minimax\\toxic_prediction\\toxic_feature_names.pkl']

In [14]:
y_col = ['herg_pIC50', 'hepg2_pIC50', 'cyp2c9_pIC50','cyp2d6_pIC50', 'cyp3a4_pIC50']

In [16]:
import numpy as np
from sklearn.datasets import load_linnerud
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor, XGBRFRegressor

# regr = MultiOutputRegressor(XGBRegressor(
#             random_state=42, n_estimators=500, min_child_weight=5, n_jobs=-1,
#             learning_rate = 0.05)).fit(x[important_col], df[y_col])

In [17]:
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
import deepchem as dc
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score

model = XGBRegressor(
            random_state=42, n_estimators=500, min_child_weight=5, n_jobs=-1,
            learning_rate = 0.5
        )

def kfold(model, X_resampled, y_resampled):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    mse_list, rmse_list, r2_list = [], [], []

    # 인덱스 리셋(정렬/정합 문제 방지)
    Xr = X_resampled.reset_index(drop=True)
    yr = y_resampled.reset_index(drop=True)
    # w_all = np.asarray(lds(yr), dtype=float)  # 전체에서 만든 가중치라면

    for tr_idx, te_idx in kf.split(Xr):
        X_train, X_test = Xr.iloc[tr_idx], Xr.iloc[te_idx]
        y_train, y_test = yr.iloc[tr_idx], yr.iloc[te_idx]
                      
        multi_model = MultiOutputRegressor(model)
        multi_model.fit(X_train, y_train)
        y_pred = multi_model.predict(X_test)
        
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        mse_list.append(mse)
        rmse_list.append(rmse)
        r2_list.append(r2)

    print("각 Fold의 MSE:", mse_list)
    print("평균 MSE:", np.mean(mse_list))
    print("각 Fold의 RMSE:", rmse_list)
    print("평균 RMSE:", np.mean(rmse_list))
    print("각 Fold의 R2:", r2_list)
    print("평균 R2:", np.mean(r2_list))

kfold(model, df_pca, df[y_col])
# pred = model.predict(ligand_df_pca)

각 Fold의 MSE: [0.16770760715007782, 0.168292835354805, 0.17147928476333618, 0.16596703231334686, 0.16995474696159363, 0.1672925055027008, 0.16549475491046906, 0.17062316834926605, 0.1739024817943573, 0.16342534124851227]
평균 MSE: 0.1684139758348465
각 Fold의 RMSE: [0.4095211925530568, 0.41023509766328503, 0.41410057324680943, 0.4073905157380899, 0.41225568153949516, 0.40901406516488015, 0.40681046558621015, 0.4130655739095986, 0.41701616490773746, 0.404259002680846]
평균 RMSE: 0.41036683329900087
각 Fold의 R2: [0.5978308320045471, 0.5917131304740906, 0.5851655006408691, 0.6003003120422363, 0.5834292769432068, 0.5907267928123474, 0.5953962206840515, 0.5802236795425415, 0.5808485746383667, 0.5933988690376282]
평균 R2: 0.5899033188819885


In [18]:
model.fit(df_pca, df[y_col])

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [19]:
import joblib
joblib.dump(model, "toxic_model.pkl")

['toxic_model.pkl']